# SQL for Data Science – Using Subqueries

This notebook covers:
- Introduction to Subqueries
- Subqueries with IN
- Aggregate Subqueries
- EXISTS
- Subqueries in FROM
- Correlated Subqueries
- Pandas Equivalent

# Load SQL Extension

In [1]:
%load_ext sql

# Connect to PostgreSQL

In [2]:
%sql postgresql://localhost/startup_commerce_db

# Create Tables

In [3]:
%%sql
CREATE TABLE customers (
    customer_id INT PRIMARY KEY,
    customer_name TEXT,
    country TEXT
);

CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    amount INT
);


 * postgresql://localhost/startup_commerce_db
Done.
(psycopg2.errors.DuplicateTable) relation "orders" already exists

[SQL: CREATE TABLE orders (
    order_id INT PRIMARY KEY,
    customer_id INT,
    amount INT
);]
(Background on this error at: https://sqlalche.me/e/20/f405)


# Insert Data

In [4]:
%%sql
INSERT INTO customers VALUES
(1,'Alice','USA'),
(2,'Bob','UK'),
(3,'Charlie','USA');

INSERT INTO orders VALUES
(101,1,500),
(102,2,300),
(103,1,700);


 * postgresql://localhost/startup_commerce_db
3 rows affected.
(psycopg2.errors.ForeignKeyViolation) insert or update on table "orders" violates foreign key constraint "orders_product_id_fkey"
DETAIL:  Key (product_id)=(500) is not present in table "products".

[SQL: INSERT INTO orders VALUES
(101,1,500),
(102,2,300),
(103,1,700);]
(Background on this error at: https://sqlalche.me/e/20/gkpj)


In [5]:
%%sql
SELECT * FROM customers;
SELECT * FROM orders;


 * postgresql://localhost/startup_commerce_db
3 rows affected.
3 rows affected.


order_id,user_id,product_id,amount,order_date
1,1,1,75000.0,2024-01-10
2,2,2,30000.0,2024-02-15
3,3,3,20000.0,2024-03-20


# Convert into Pandas

In [6]:
import pandas as pd

customers_df = pd.DataFrame({
    'customer_id':[1,2,3],
    'customer_name':['Alice','Bob','Charlie'],
    'country':['USA','UK','USA']
})

orders_df = pd.DataFrame({
    'order_id':[101,102,103],
    'customer_id':[1,2,1],
    'amount':[500,300,700]
})


# Subquery with IN

In [7]:
%%sql
SELECT customer_name
FROM customers
WHERE customer_id IN (
    SELECT customer_id
    FROM orders
);


 * postgresql://localhost/startup_commerce_db
3 rows affected.


customer_name
Alice
Bob
Charlie


In [8]:
customers_df[customers_df['customer_id'].isin(orders_df['customer_id'])]

,customer_id,customer_name,country
0,1,Alice,USA
1,2,Bob,UK


# Aggregate Subquery

In [9]:
%%sql
SELECT *
FROM orders
WHERE amount >
(
    SELECT AVG(amount)
    FROM orders
);


 * postgresql://localhost/startup_commerce_db
1 rows affected.


order_id,user_id,product_id,amount,order_date
1,1,1,75000.0,2024-01-10


In [10]:
orders_df[orders_df['amount'] > orders_df['amount'].mean()]

,order_id,customer_id,amount
2,103,1,700


# EXISTS Operator

In [21]:
%%sql
SELECT name
FROM users u
WHERE EXISTS (
    SELECT 1
    FROM orders o
    WHERE o.user_id = u.user_id
);

 * postgresql://localhost/startup_commerce_db
3 rows affected.


name
Alice
Bob
charlie


# Subquery in FROM Clause

In [22]:
%%sql
SELECT *
FROM (
    SELECT user_id,
           SUM(amount) AS total_amount
    FROM orders
    GROUP BY user_id
) AS customer_totals;

 * postgresql://localhost/startup_commerce_db
3 rows affected.


user_id,total_amount
3,20000.0
2,30000.0
1,75000.0


# Correlated Subquery

In [23]:
%%sql
SELECT o1.*
FROM orders o1
WHERE o1.amount >
(
    SELECT AVG(o2.amount)
    FROM orders o2
    WHERE o1.user_id = o2.user_id
);

 * postgresql://localhost/startup_commerce_db
0 rows affected.


order_id,user_id,product_id,amount,order_date


In [14]:
orders_df.groupby('customer_id')['amount'].transform('mean')

0    600.0
1    300.0
2    600.0
Name: amount, dtype: float64

# Practice Exercise

In [15]:
%%sql
SELECT customer_name
FROM customers
WHERE customer_id NOT IN (
    SELECT customer_id
    FROM orders
);


 * postgresql://localhost/startup_commerce_db
0 rows affected.


customer_name


In [19]:
%%sql
SELECT *
FROM users
LIMIT 5;

 * postgresql://localhost/startup_commerce_db
4 rows affected.


user_id,name,email,city
1,Alice,ali@gmail.com,Delhi
2,Bob,None,Mumbai
3,charlie,charlie@gmail.com,Bangalore
4,David,None,Delhi
